# 噴塗系統報告
## 數據前處理與邊緣端整合規劃 

本報告說明在智慧噴塗產線（SprayLine）專案中，資料前處理管線（Data Preprocessing Pipeline）的系統架構定位、輸入規格、核心演算法設計以及輸出資料流，以確保後續資料庫關聯與機器學習模型的數據品質。

--- 
### 核心系統架構與分工定位
在整體的工業物聯網（IIoT）與多代理人（Multi-Agent）架構中，本站負責的**資料前處理服務**位居 **Edge Layer（邊緣層）與 Data Layer（數據層）的交界關鍵**。

為確保系統穩定性與前後端解耦，本站的系統透過REST API負責接收底層各 Agent（如 Spraying Agent-1）採集到的原始物理感測器訊號。由於原始數據存在高噪聲、時間不同步與斷線遺失等問題，必須先交由本站進行數據清洗，才能接續給後續團隊：

1. **輸入 (Input)**：提供 API Endpoint，接收來自邊緣服務的 JSON 批次資料。
2. **核心 (Kernel)**：執行 API 封包解析、缺失值填補、突波過濾與時序訊號平滑化。
3. **輸出 (Output)**：提供結構化、無噪聲的乾淨數據（Clean Data）供下一站寫入 SQL 資料庫，並供其他站進行特徵提取與剩餘壽命（RUL）預測。

## 1. 輸入階段 (Input) — REST API 接收端
底層 `Chamber_1` 內的邊緣代理人，會將高頻採集的動態數據打包成 JSON 格式，並定時（例如每秒或每個加工週期）透過 `HTTP POST` 請求，發送至本站負責的 API 進入點。

預期接收的欄位清單如下：

** 生產履歷與系統資訊 (Traceability & System Info)**
* `line_id`: 產線編號 
* `batch_id`: 批次編號 
* `part_id`: 零件編號 
* `process_stage`: 製程階段 
* `chamber_id`: 腔體編號 
* `timestamp`: 資料產生的精確時間戳記

** 流體動力感測 (Fluid & Power)**
* `pump_current`: 幫浦馬達電流 
* `spray_pressure`: 噴塗壓力
* `spray_flow_rate`: 噴塗流速/流量

** 運動學與控制器 (Kinematics & Controller)**
* `path_file`: 正在執行的配方路徑檔名
* `path_block`: 當前執行的程式碼行號/區塊進度
* `nozzle_pos_x`, `nozzle_pos_y`, `nozzle_pos_z`: 噴嘴三維絕對座標
* `nozzle_roll`, `nozzle_pitch`, `nozzle_yaw`: 噴嘴翻滾、俯仰、偏航姿態角度
* `nozzle_vel_x`, `nozzle_vel_y`, `nozzle_vel_z`: 噴嘴移動速度

** 環境與耗材 (Environment & Consumables)**
* `filter_inflow`: 濾網進氣/進液流量 
* `filter_outflow`: 濾網出氣/出液流量 
* `chamber_temperature`: 腔體內環境溫度
* `chamber_humidity`: 腔體內環境濕度
* `material`: 正在使用的塗料/材質型號
* `spray_width`: CCD 視覺即時回傳的噴幅寬度

## 2. 核心處理階段 (Kernel) — 前處理管線 (Preprocessing Pipeline)
當 API 接收到 JSON 封包並驗證無誤後，會立即轉換為 DataFrame，並啟動以下四道高效能清洗管線：

1. **時間戳記對齊 (Timestamp Alignment)**：解析 Edge 端回傳的時間，將異步採集的各感測器時間戳重新採樣（Resampling）至統一頻率，建立標準時間軸。
2. **缺失值處理 (Missing Value Imputation)**：針對工業網路斷線導致的遺失值，採用**前向填充（Forward Fill）**結合**線性插值（Linear Interpolation）**，避免破壞物理時間的連續性。
3. **異常值過濾 (Outlier Detection & Filtering)**：利用 **IQR (四分位距法)** 或 **Z-Score** 鎖定感測器因電磁干擾產生的非物理突波（例如不合理的極端電流），並將其平滑化或剔除。
4. **信號平滑化 (Signal Smoothing)**：針對高頻微幅震盪的電流與壓力訊號，引入**滑動平均濾波器（Rolling Mean）**，提取出核心製程特徵，過濾環境雜訊。

## 3. 輸出階段 (Output) — Clean Data
經過前處理核心清洗後的數據，將直接驅動後續的系統應用，具備三大核心價值：

1. **交棒給其它站(DB/SQL Team)**：輸出格式為欄位標準化、時間軸對齊且無 `NaN` 的資料流，可直接執行高效能資料庫 `INSERT`，確保 Schema 寫入不會報錯。
2. **交棒給其它站(Analytics & AI Team)**：清洗後的平滑特徵，讓後續的 Past/Current/Future 服務能基於真實物理特徵，更精準地計算設備剩餘壽命（RUL）或分析腔體稼動率（Utilization %）。
3. **交棒給其它站(UI/Visual Team)**：確保 Omniverse 前端 3D 視覺化在繪製歷史與即時趨勢圖時，線條平穩，不會出現因異常突波造成的畫面劇烈跳動。